# Example in Section III-B

The details of the example in Section III-B.

In [1]:
import sys
sys.path.append('../')
import copy
from net.get_net import get_net_examle
import torch
from easy_input.get_input import EasyProperty, EasyOutConstr
from relational_property.relational_analysis import RelationalAnalysis
from relational_split.rs_back import RSInfo
from individual_split.is_back import IS_info
from relational_bounds.relational_back_substitution import IndividualAndRelationalBounds
from IPython.display import Math, display

## The approximate bounds derived from the symbolic bound propagation in RaVeN

You can get the same bounds in the paper (Example 1) by running the notebook below.

<figure>
    <img src="bounds.png" alt="Fig. 2-(a): Approximate symbolic bounds propagation" width="400"/>
    <figcaption>Fig. 1: Approximate symbolic bounds propagation</figcaption>
</figure>

This figure (same as Fig. 2-(a) in the paper) shows the propagated bounds for individual and relational neurons derived from the backpropagation based on the symbolic bound propagation in RaVeN.

## Process for getting the initial bounds by backsubstitution
1. Define the network, input, and properties
2. Run backsubstitution to get initial bounds
    - `run` returns individual bounds (`inp1_lbs`, `inp1_ubs`, `inp2_lbs`, `inp2_ubs`) and relational bounds (`d_lbs`, `d_ubs`) 

In [2]:
# Define the network, input, and properties (dummy example)
net = get_net_examle()
inp = torch.tensor([0.2, 0.3])
epsilon = torch.tensor([0.1, 0.1])
delta = 0.05
inp_lb = (inp - epsilon).clamp(min=0.0)
inp_ub = (inp + epsilon).clamp(max=1.0)
target_dim = 0
out_constr = EasyOutConstr(target_dim)
inp1_prop = EasyProperty(inp, inp_lb, inp_ub, out_constr)
inp2_prop = EasyProperty(inp, inp_lb, inp_ub, out_constr)
layer_names = ["y^(0)", "x^(1)", "y^(1)", "x^(2)"]
neuron_names = [["y (inp1)", "hat y (inp2)", "Delta y (relation)"], 
                ["x (inp1)", "hat x (inp2)", "Delta x (relation)"], 
                ["y (inp1)", "hat y (inp2)", "Delta y (relation)"], 
                ["x (inp1)", "hat x (inp2)", "Delta x (relation)"]]

# run backsubstitution to get initial bounds
print("-- Initial bounds (Backsubstitution) --")
irb = IndividualAndRelationalBounds(inp1_prop, inp2_prop, net, None, delta, device='cpu', refine_bounds_prop=True, backprop_mode='DP')
inp1_lbs, inp1_ubs, inp2_lbs, inp2_ubs, d_lbs, d_ubs = irb.run(logging=False)
for i in range(len(inp1_lbs)):
    print(f"Layer {layer_names[i]}:")
    inp1_neuron = neuron_names[i][0]
    inp2_neuron = neuron_names[i][1]
    d_neuron = neuron_names[i][2]
    for j in range(len(inp1_lbs[i])):
        print(f"  Position {j+1}:")
        print(f"    {inp1_neuron} in [{inp1_lbs[i][j]:.6f}, {inp1_ubs[i][j]:.6f}]")
        print(f"    {inp2_neuron} in [{inp2_lbs[i][j]:.6f}, {inp2_ubs[i][j]:.6f}]")
        print(f"    {d_neuron} in [{d_lbs[i][j]:.6f}, {d_ubs[i][j]:.6f}]")
    print("")

-- Initial bounds (Backsubstitution) --
Layer y^(0):
  Position 1:
    y (inp1) in [0.100000, 0.300000]
    hat y (inp2) in [0.100000, 0.300000]
    Delta y (relation) in [-0.050000, 0.050000]
  Position 2:
    y (inp1) in [0.200000, 0.400000]
    hat y (inp2) in [0.200000, 0.400000]
    Delta y (relation) in [-0.050000, 0.050000]

Layer x^(1):
  Position 1:
    x (inp1) in [-0.300000, 0.100000]
    hat x (inp2) in [-0.300000, 0.100000]
    Delta x (relation) in [-0.100000, 0.100000]
  Position 2:
    x (inp1) in [-0.400000, 0.200000]
    hat x (inp2) in [-0.400000, 0.200000]
    Delta x (relation) in [-0.150000, 0.150000]

Layer y^(1):
  Position 1:
    y (inp1) in [0.000000, 0.100000]
    hat y (inp2) in [0.000000, 0.100000]
    Delta y (relation) in [-0.100000, 0.100000]
  Position 2:
    y (inp1) in [0.000000, 0.200000]
    hat y (inp2) in [0.000000, 0.200000]
    Delta y (relation) in [-0.200000, 0.200000]

Layer x^(2):
  Position 1:
    x (inp1) in [-0.200000, 0.100000]
    hat x

## The splitting analysis

We show the performance difference between individual and relational splitting.

- Individual splitting performs at the individual neron $x^{(1)}_1$.
- Relational splitting performs at the relational neuron $\Delta x^{(1)}_1$.  (Both neurons are colored in green in the figure above.)

<img src="splitting.png" alt="Fig. 2-(b): Splitting of individual neurons vs. relational neurons" width="600"/>

This figure (same as Fig. 2-(b): Splitting of individual neurons vs. relational neurons) shows the splitting results for individual and relational splitting. 
You can get the same splitting process and results as shown in this figure by running the notebook below.


### Procedure to generate the splitting results
1. build the LP model without any splits
    - `create_lp_model`
2. duplicate the LP model for each subproblem
    - `duplicate`
3. define the split information (e.g., which neuron to split, individual or relational splitting)
    - `IS_info` (individual) and `RS_info` (relational)
4. add the split constraints to the duplicated model based on the split information
    - `update_lp_model`
5. solve the LP for each subproblem
    - `minimize_lp` and `maximize_lp`

As a result, Individual splitting (IS) and relational splitting (RS) give two subproblems each. Subproblem 1 and subproblem 2 correspond to the negative (e.g., $x^{(1)}_1 \leq 0$) and positive branches (e.g., $x^{(1)}_1 \geq 0$) after splitting, respectively.

By union the results of the two subproblems, we can get the final bounds after splitting.

In [4]:
# LP without any splits
root_lpr = RelationalAnalysis()
root_lpr.grb_model = root_lpr.create_lp_model(net, irb.shapes, inp1_lbs, inp1_ubs, inp2_lbs, inp2_ubs, d_lbs, d_ubs)

out_delta = 'Delta x (2,1)'

print("\n-- Splitting analysis --")
# split at layer 1, position 1
split_pos = 0  # 0-based index
print(f"Split at layer 1, position {split_pos+1}")


print("\n* Individual Split *")
split_info = IS_info('ISA', 1, split_pos, 0.0)
split_info1 = copy.deepcopy(split_info)
split_info1.is_type = f"{split_info1.is_inp}1"
lpr1_is = RelationalAnalysis()
lpr1_is.duplicate(root_lpr)
lpr1_is.update_lp_model(lpr1_is.grb_model, None, [split_info1], inp1_lbs, inp1_ubs, inp2_lbs, inp2_ubs, d_lbs, d_ubs)
split_info2 = copy.deepcopy(split_info)
split_info2.is_type = f"{split_info2.is_inp}2"
lpr2_is = RelationalAnalysis()
lpr2_is.duplicate(root_lpr)
lpr2_is.update_lp_model(lpr2_is.grb_model, None, [split_info2], inp1_lbs, inp1_ubs, inp2_lbs, inp2_ubs, d_lbs, d_ubs)
print(f"output delta after splitting individual neuron x ({split_info.layer_idx},{split_info.pos+1})")
d_lb_i1 = lpr1_is.minimize_lp(lpr1_is.grb_model, lpr1_is.grb_delta_vars[-1][0])
d_ub_i1 = lpr1_is.maximize_lp(lpr1_is.grb_model, lpr1_is.grb_delta_vars[-1][0])
print(f"  Subproblem 1 with split constraint (x (1,{split_pos+1}) <= 0): {out_delta} in [{d_lb_i1:.6f}, {d_ub_i1:.6f}]")
d_lb_i2 = lpr2_is.minimize_lp(lpr2_is.grb_model, lpr2_is.grb_delta_vars[-1][0])
d_ub_i2 = lpr2_is.maximize_lp(lpr2_is.grb_model, lpr2_is.grb_delta_vars[-1][0])
print(f"  Subproblem 2 with split constraint (x (1,{split_pos+1}) >= 0): {out_delta} in [{d_lb_i2:.6f}, {d_ub_i2:.6f}]")
print(f"  Union: {out_delta} in [{min(d_lb_i1, d_lb_i2):.6f}, {max(d_ub_i1, d_ub_i2):.6f}]")


print("\n* Relational Split *")
split_info = RSInfo('RSZ', 1, split_pos, 0.0)
split_info1 = copy.deepcopy(split_info)
split_info1.rs_type = f"{split_info1.rs_type}1"
lpr1_rs = RelationalAnalysis()
lpr1_rs.duplicate(root_lpr)
lpr1_rs.update_lp_model(lpr1_rs.grb_model, [split_info1], None, inp1_lbs, inp1_ubs, inp2_lbs, inp2_ubs, d_lbs, d_ubs)
split_info2 = copy.deepcopy(split_info)
split_info2.rs_type = f"{split_info2.rs_type}2"
lpr2_rs = RelationalAnalysis()
lpr2_rs.duplicate(root_lpr)
lpr2_rs.update_lp_model(lpr2_rs.grb_model, [split_info2], None, inp1_lbs, inp1_ubs, inp2_lbs, inp2_ubs, d_lbs, d_ubs)
print(f"output delta after splitting relational neuron Delta x ({split_info.layer_idx},{split_info.pos+1})")
d_lb_r1 = lpr1_rs.minimize_lp(lpr1_rs.grb_model, lpr1_rs.grb_delta_vars[-1][0])
d_ub_r1 = lpr1_rs.maximize_lp(lpr1_rs.grb_model, lpr1_rs.grb_delta_vars[-1][0])
print(f"  Subproblem 1 with split constraint (Delta x (1,{split_pos+1}) <= 0): {out_delta} in [{d_lb_r1:.6f}, {d_ub_r1:.6f}]")
d_lb_r2 = lpr2_rs.minimize_lp(lpr2_rs.grb_model, lpr2_rs.grb_delta_vars[-1][0])
d_ub_r2 = lpr2_rs.maximize_lp(lpr2_rs.grb_model, lpr2_rs.grb_delta_vars[-1][0])
print(f"  Subproblem 2 with split constraint (Delta x (1,{split_pos+1}) >= 0): {out_delta} in [{d_lb_r2:.6f}, {d_ub_r2:.6f}]")
print(f"  Union: {out_delta} in [{min(d_lb_r1, d_lb_r2):.6f}, {max(d_ub_r1, d_ub_r2):.6f}]")


-- Splitting analysis --
Split at layer 1, position 1

* Individual Split *
output delta after splitting individual neuron x (1,1)
  Subproblem 1 with split constraint (x (1,1) <= 0): Delta x (2,1) in [-0.150000, 0.066667]
  Subproblem 2 with split constraint (x (1,1) >= 0): Delta x (2,1) in [-0.150000, 0.149999]
  Union: Delta x (2,1) in [-0.150000, 0.149999]

* Relational Split *
output delta after splitting relational neuron Delta x (1,1)
  Subproblem 1 with split constraint (Delta x (1,1) <= 0): Delta x (2,1) in [-0.066667, 0.066667]
  Subproblem 2 with split constraint (Delta x (1,1) >= 0): Delta x (2,1) in [-0.066667, 0.150000]
  Union: Delta x (2,1) in [-0.066667, 0.150000]
